In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [17]:
train_df= pd.read_csv("train.csv")
test_df= pd.read_csv("test.csv")

In [18]:
from torchvision import transforms

train_transforms= transforms.Compose([
    transforms.ColorJitter(0.2, 0.2),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

eval_transforms= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [19]:
from torch.utils.data import Dataset, DataLoader
import torch

class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df= df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row= self.df.iloc[idx]
        img_path= row['ImagePath']
        img= Image.open(img_path)
        if(img.mode == "P"):
            img= img.convert('RGBA')
        img= img.convert('RGB')

        if self.transform:
            img = self.transform(img)
        
        if 'Label' not in row:
            return img
        
        label= torch.tensor(row["Label"], dtype=torch.long)
        return img, label

In [20]:
train_dataset= ImageDataset(train_df, transform= train_transforms)
test_dataset= ImageDataset(test_df, transform= eval_transforms)

train_loader= DataLoader(train_dataset, batch_size=32, shuffle= True)
test_loader= DataLoader(test_dataset, batch_size=32, shuffle= False)

In [21]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

model= efficientnet_b0(weights= EfficientNet_B0_Weights.DEFAULT)
model.classifier[1]= nn.Linear(model.classifier[1].in_features, 2)

In [22]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'

epochs= 5
lr= 1e-5

criterion= nn.CrossEntropyLoss()
optimizer= torch.optim.AdamW(model.parameters(), lr= lr)

model.to(device)

for epoch in range(epochs):
    model.train()
    running_loss= 0.0
    for images, labels in train_loader:
        images, labels= images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs= model(images)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss+= loss.item()
    running_loss/= len(train_loader)
    print(f"Epoch:{epoch+ 1}, Loss:{running_loss}")

Epoch:1, Loss:0.6888417673110961
Epoch:2, Loss:0.6645527911186219
Epoch:3, Loss:0.6454215669631957
Epoch:4, Loss:0.6339788484573364
Epoch:5, Loss:0.6176624608039856


In [23]:
predictions=[]

with torch.no_grad():
    for images in test_loader:
        images= images.to(device)

        outputs= model(images)

        preds= torch.argmax(outputs, 1)
        predictions.extend(preds.cpu().numpy())

answer= []

for idx, row in test_df.iterrows():
    answer.append({
        "SampleID": row['SampleID'],
        "Label": predictions[idx]
    })

pd.DataFrame(answer).to_csv("submission.csv", index= False)